In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import NaturalEarthFeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER
import glob
import re
import intake
from pathlib import Path

%matplotlib inline

In [ ]:
from dask.distributed import Client

client = Client(threads_per_worker=1)
client

In [ ]:
ds_input = xr.open_dataset('/scratch/vn19/nm5072/thresholds_tmp/OM2_01deg_jra55v140_iaf_WA_2011_rechunked.zarr').temp

In [ ]:
ds_input

### load and concatenate threshold and seasonal cycle

In [ ]:
# Load seasonal cycle files
files = sorted(input_dir.glob("OM2_01deg_jra55v140_iaf_WA_depth*_seas_rechunked.nc"))

datasets = []
for f in files:
    ds = xr.open_dataset(f)
    depth_value = ds["st_ocean"].values.item()
    ds = ds.expand_dims("st_ocean").assign_coords({"st_ocean": [depth_value]})
    datasets.append(ds)

seas = xr.concat(datasets, dim="st_ocean").sortby("st_ocean")

# --- FIX: Rename dimensions right after loading if they are incorrect ---
if 'lat' in seas.dims and 'lon' in seas.dims:
    seas = seas.rename({'lat': 'yt_ocean', 'lon': 'xt_ocean'})

In [ ]:
# Load threshold files
files = sorted(input_dir.glob("OM2_01deg_jra55v140_iaf_WA_depth*_thresh_rechunked.nc"))

datasets = []
for f in files:
    ds = xr.open_dataset(f)
    # NOTE: It's good practice to give your data variable a name when saving,
    # rather than letting xarray default to '__xarray_dataarray_variable__'.
    # For now, we will work with the default name.
    ds = ds.rename({'__xarray_dataarray_variable__': 'thresh'})
    depth_value = ds["st_ocean"].values.item()
    ds = ds.expand_dims("st_ocean").assign_coords({"st_ocean": [depth_value]})
    datasets.append(ds)

thresh = xr.concat(datasets, dim="st_ocean").sortby("st_ocean")

# --- FIX: Rename dimensions right after loading if they are incorrect ---
if 'lat' in thresh.dims and 'lon' in thresh.dims:
    thresh = thresh.rename({'lat': 'yt_ocean', 'lon': 'xt_ocean'})

In [ ]:
#  Align depth levels to a common set
nlev_seas = seas.sizes["st_ocean"]
nlev_thresh = thresh.sizes["st_ocean"]
nlev_common = min(nlev_seas, nlev_thresh)

seas = seas.isel(st_ocean=slice(nlev_common))
thresh = thresh.isel(st_ocean=slice(nlev_common))
ds_input = ds_input.isel(st_ocean=slice(nlev_common))


# 2) Assign dayofyear to ds_input based on its 'time'
ds_input = ds_input.assign_coords(dayofyear=ds_input['time'].dt.dayofyear)

# 3) Align Seas and Thresh to ds_input's dayofyear. This remains lazy.
# The 'dayofyear' coordinate will broadcast correctly during the operation.
seas_aligned = seas.sel(dayofyear=ds_input.dayofyear)
thresh_aligned = thresh.sel(dayofyear=ds_input.dayofyear)

# 4) Calculate anomaly and severity. The results are still lazy Dask arrays.
# This builds the computation graph.
SSTa = ds_input - seas_aligned.temp
Severity = SSTa / (thresh_aligned.thresh - seas_aligned.temp)

In [ ]:
# 1) Combine into a new Dataset
Severity_ds = xr.Dataset({
    'ssta': SSTa,
    'severity': Severity
})

In [ ]:
zarr_filename = output_dir / f"OM2_{experiment}_WA_2011_severity_rechunked.zarr"
print(f"\\nWriting out Zarr to: {zarr_filename}")
Severity_ds.to_zarr(zarr_filename, mode='w', consolidated=True)

In [ ]:
Severity_ds

In [ ]:
Severity_ds.severity.isel(time=1,st_ocean=1).plot()